In [ ]:
import numpy as np

from tardis_em.cnn.data_processing.draw_mask import draw_instances
from tardis_em_analysis.stitch_volume.align_tomograms import VolumeRidgeRegistration
from tardis_em.utils.load_data import load_image, ImportDataFromAmira

am = ImportDataFromAmira('/Users/robertkiewisz/Downloads/C.elegans_MaleMeiosis/T0391_worm13_metaphase02_4_spatialGraph.am', '/Users/robertkiewisz/Downloads/C.elegans_MaleMeiosis/T0391_worm13_metaphase02_4.am')
img1, px = am.get_image()
am = ImportDataFromAmira('/Users/robertkiewisz/Downloads/C.elegans_MaleMeiosis/T0391_worm13_metaphase02_5_spatialGraph.am', '/Users/robertkiewisz/Downloads/C.elegans_MaleMeiosis/T0391_worm13_metaphase02_5.am')
img2, px = am.get_image()

## Synthetic Validation — verify methods work on a known transform
Apply a known rotation + translation to one projection, then check if each method recovers the parameters.

In [ ]:
aligner = VolumeRidgeRegistration('akaze', 10)

vol2_aligned = aligner(img1, img2, return_aligned=False)
print(vol2_aligned)

img_p_sift = aligner.apply_rigid_transform(
    img2[0], 
    angle=vol2_aligned['Angle'], 
    tx=vol2_aligned['Tx']/10, 
    ty=vol2_aligned['Ty']/10, 
    scale=vol2_aligned['Scale']
    )
show_image_list([img1[-1], img_p_sift, np.std([img1[-1], img_p_sift], axis=0)], num_cols=3)

## SimpleITK — Similarity2D + Mattes MI + Multi-Resolution

In [ ]:
aligner = VolumeRidgeRegistration('sitk', down_scale=10)

vol2_aligned = aligner(img1, img2, return_aligned=False)
print("SimpleITK result:", vol2_aligned)

img_p_sitk = aligner.apply_rigid_transform(
    img2[0], 
    angle=vol2_aligned['Angle'], 
    tx=vol2_aligned['Tx'], 
    ty=vol2_aligned['Ty'], 
    scale=vol2_aligned['Scale']
)


In [ ]:
from tardis_em.utils.visualize_pc import show_image_list
show_image_list([img1[-1], img2[0], np.std([img1[-1], img2[0]], axis=0)], num_cols=3)
show_image_list([img1[-1], img_p_sitk, np.std([img1[-1], img_p_sitk], axis=0)], num_cols=3)


## pyStackReg — SCALED_ROTATION (TurboReg/StackReg port)

In [ ]:
aligner = VolumeRidgeRegistration('stackreg', down_scale=10)

vol2_aligned = aligner(img1, img2, return_aligned=False)
print("pyStackReg result:", vol2_aligned)

img_p_stackreg = aligner.apply_rigid_transform(
    img2[0], 
    angle=vol2_aligned['Angle'], 
    tx=vol2_aligned['Tx'], 
    ty=vol2_aligned['Ty'], 
    scale=vol2_aligned['Scale']
)


In [ ]:
show_image_list([img1[-1], img2[0], np.std([img1[-1], img2[0]], axis=0)], num_cols=3)
show_image_list([img1[-1], img_p_stackreg, np.std([img1[-1], img_p_stackreg], axis=0)], num_cols=3)


## Log-Polar + Phase Correlation (scikit-image)

In [ ]:
aligner = VolumeRidgeRegistration('logpolar', down_scale=10)

vol2_aligned = aligner(img1, img2, return_aligned=False)
print("Log-polar result:", vol2_aligned)

img_p_logpolar = aligner.apply_rigid_transform(
    img2[0], 
    angle=vol2_aligned['Angle'], 
    tx=vol2_aligned['Tx'], 
    ty=vol2_aligned['Ty'], 
    scale=vol2_aligned['Scale']
)


In [ ]:
show_image_list([img1[-1], img2[0], np.std([img1[-1], img2[0]], axis=0)], num_cols=3)
show_image_list([img1[-1], img_p_logpolar, np.std([img1[-1], img_p_logpolar], axis=0)], num_cols=3)


## imreg_dft — FFT-based Similarity Registration (Reddy & Chatterji 1996)

In [ ]:
aligner = VolumeRidgeRegistration('imregdft', down_scale=10)

vol2_aligned = aligner(img1, img2, return_aligned=False)
print("imreg_dft result:", vol2_aligned)

img_p_imregdft = aligner.apply_rigid_transform(
    img2[0], 
    angle=vol2_aligned['Angle'], 
    tx=vol2_aligned['Tx'], 
    ty=vol2_aligned['Ty'], 
    scale=vol2_aligned['Scale']
)


In [ ]:
show_image_list([img1[-1], img2[0], np.std([img1[-1], img2[0]], axis=0)], num_cols=3)
show_image_list([img1[-1], img_p_imregdft, np.std([img1[-1], img_p_imregdft], axis=0)], num_cols=3)


In [ ]:
from tardis_em_analysis.stitch_volume.align_tomograms import stitch_tomogram_stack
stitch_tomogram_stack(
    input_dir='/Users/robertkiewisz/Downloads/C.elegans_FemalePN/',
    output_dir='/Users/robertkiewisz/Downloads/C.elegans_FemalePN/stitched_output',
    method='mt',
)

2026-03-29 21:16:38 - tardis_em - INFO - Found 5 tomograms in /Users/robertkiewisz/Downloads/C.elegans_FemalePN/
2026-03-29 21:16:39 - tardis_em - INFO - Registering pair 0 → 1 ...
2026-03-29 21:16:39 - tardis_em - INFO - MT matching: 41/108 endpoints matched  (score=0.380)
2026-03-29 21:16:39 - tardis_em - INFO - Pair 0→1: Angle=-0.82  Tx=-4.3  Ty=-5.9  Scale=1.0402  Score=0.3796
2026-03-29 21:16:39 - tardis_em - INFO - Registering pair 1 → 2 ...
2026-03-29 21:16:39 - tardis_em - INFO - MT matching: 46/106 endpoints matched  (score=0.434)
2026-03-29 21:16:39 - tardis_em - INFO - Pair 1→2: Angle=0.52  Tx=13.4  Ty=19.2  Scale=0.9666  Score=0.4340
2026-03-29 21:16:39 - tardis_em - INFO - Registering pair 2 → 3 ...
2026-03-29 21:16:39 - tardis_em - INFO - MT matching: 65/106 endpoints matched  (score=0.613)
2026-03-29 21:16:39 - tardis_em - INFO - Pair 2→3: Angle=0.63  Tx=-2.2  Ty=2.5  Scale=1.0082  Score=0.6132
2026-03-29 21:16:39 - tardis_em - INFO - Registering pair 3 → 4 ...
2026-03-2